# Homework 1 — File Reading Agent with LangChain Tools

Build an agent that can **read a file** and **count its characters** using LangChain tools.

| Step | Description |
|------|-------------|
| 1 | Create a sample `.txt` file to work with |
| 2 | Define `read_file` and `get_content_length` tools |
| 3 | Bind tools to the LLM |
| 4 | Build the agent loop — model calls tools automatically |
| 5 | Try more questions on the file |

In [1]:
!pip install langchain langchain-ollama --quiet

## Cell 1 — Create a Sample .txt File

In [1]:
sample_content = """Welcome to the AI Training Course.

This course covers the fundamentals of large language models,
prompt engineering, RAG pipelines, and building AI-powered applications.

Topics include:
- Tokenization and embeddings
- LangChain tools and agents
- Vector stores and retrieval
- Conversational AI with memory

We hope you enjoy the journey into generative AI!
"""

with open('sample.txt', 'w') as f:
    f.write(sample_content)

print('sample.txt created!')
print(f'Content preview:\n{sample_content[:100]}...')

sample.txt created!
Content preview:
Welcome to the AI Training Course.

This course covers the fundamentals of large language models,
pr...


## Cell 2 — Define Tools

In [2]:
from langchain_core.tools import tool

@tool
def read_file(filename: str) -> str:
    """Reads the contents of a text file and returns the content as a string."""
    try:
        with open(filename, 'r') as f:
            content = f.read()
        return content
    except FileNotFoundError:
        return f"Error: File '{filename}' not found."

@tool
def get_content_length(filename: str) -> str:
    """Returns the number of characters in a text file. Pass the filename to count its characters."""
    try:
        with open(filename, 'r') as f:
            content = f.read()
        return f"The file '{filename}' contains {len(content)} characters."
    except FileNotFoundError:
        return f"Error: File '{filename}' not found."

# Verify tools
print('Tool 1:', read_file.name, '-', read_file.description)
print('Tool 2:', get_content_length.name, '-', get_content_length.description)

# Quick direct test
content = read_file.invoke({'filename': 'sample.txt'})
length = get_content_length.invoke({'filename': 'sample.txt'})
print(f'\nDirect test:\n{length}')


Tool 1: read_file - Reads the contents of a text file and returns the content as a string.
Tool 2: get_content_length - Returns the number of characters in a text file. Pass the filename to count its characters.

Direct test:
The file 'sample.txt' contains 360 characters.


## Cell 3 — Bind Tools to the LLM

In [3]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model='llama3.1', temperature=0)
tools = [read_file, get_content_length]
tool_map = {t.name: t for t in tools}

llm_with_tools = llm.bind_tools(tools)

print('LLM bound with tools:', [t.name for t in tools])

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


LLM bound with tools: ['read_file', 'get_content_length']


## Cell 4 — Build the Agent Loop

The agent loop:
1. Send user message to model
2. If model requests a tool → execute it → send result back
3. Repeat until model gives a final text answer

In [4]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

SYSTEM_PROMPT = """You are a helpful file assistant.
Use the tools to answer questions about files accurately."""

def run_agent(user_message: str) -> str:
    """Run the agent loop for a single user message."""
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_message)
    ]
    
    print(f'User: {user_message}')
    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        
        # No tool calls — final answer
        if not response.tool_calls:
            print(f'Agent: {response.content}\n')
            return response.content

        print(response.tool_calls)
        
        # Execute each tool the model requested
        for tool_call in response.tool_calls:
            print(f'  -> Calling tool: {tool_call["name"]} with args {tool_call["args"]}')
            tool_fn = tool_map[tool_call['name']]
            tool_result = tool_fn.invoke(tool_call['args'])
            messages.append(
                ToolMessage(content=str(tool_result), tool_call_id=tool_call['id'])
            )

    print('Agent loop ready!')

## Cell 5 — Test the Agent

Example flow:
```
"How many characters in sample.txt?"
  → calls read_file('sample.txt')
  → calls get_content_length(content)
  → "The file contains 323 characters."
```

In [5]:
# Test 1 — character count
run_agent("read file sample.txt and give me file content length?")

User: read file sample.txt and give me file content length?
[{'name': 'read_file', 'args': {'filename': 'sample.txt'}, 'id': '3c7b0d20-d0b8-48bd-a6ac-9b6711795c23', 'type': 'tool_call'}, {'name': 'get_content_length', 'args': {'filename': 'sample.txt'}, 'id': 'd22589c8-6a07-4a20-a476-c4bc69be4319', 'type': 'tool_call'}]
  -> Calling tool: read_file with args {'filename': 'sample.txt'}
  -> Calling tool: get_content_length with args {'filename': 'sample.txt'}
Agent: The file 'sample.txt' contains 360 characters.



"The file 'sample.txt' contains 360 characters."

In [6]:
# Test 2 — read content
run_agent("What topics are covered in sample.txt?")

User: What topics are covered in sample.txt?
[{'name': 'read_file', 'args': {'filename': 'sample.txt'}, 'id': '4225f86b-f226-47d6-a197-5b1f6d4a5540', 'type': 'tool_call'}]
  -> Calling tool: read_file with args {'filename': 'sample.txt'}
Agent: The output from the tool call indicates that the file "sample.txt" contains information about the topics covered in the AI Training Course. The topics include tokenization and embeddings, LangChain tools and agents, vector stores and retrieval, and conversational AI with memory.



'The output from the tool call indicates that the file "sample.txt" contains information about the topics covered in the AI Training Course. The topics include tokenization and embeddings, LangChain tools and agents, vector stores and retrieval, and conversational AI with memory.'

In [7]:
# Test 3 — combine both tools
run_agent("Read sample.txt and tell me both its content summary and character count.")

User: Read sample.txt and tell me both its content summary and character count.
[{'name': 'read_file', 'args': {'filename': 'sample.txt'}, 'id': '6209cda3-fe0c-42ea-93cb-50b0f5e86fc0', 'type': 'tool_call'}, {'name': 'get_content_length', 'args': {'filename': 'sample.txt'}, 'id': '215eb942-cb4c-4179-ac06-b11290df635c', 'type': 'tool_call'}]
  -> Calling tool: read_file with args {'filename': 'sample.txt'}
  -> Calling tool: get_content_length with args {'filename': 'sample.txt'}
Agent: The file 'sample.txt' contains a text about the AI Training Course, which is a comprehensive guide to large language models, prompt engineering, RAG pipelines, and building AI-powered applications. The content summary is a brief overview of the course topics, including tokenization and embeddings, LangChain tools and agents, vector stores and retrieval, and conversational AI with memory.



"The file 'sample.txt' contains a text about the AI Training Course, which is a comprehensive guide to large language models, prompt engineering, RAG pipelines, and building AI-powered applications. The content summary is a brief overview of the course topics, including tokenization and embeddings, LangChain tools and agents, vector stores and retrieval, and conversational AI with memory."